In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 1: CONFIGURATION AND SETUP
# ENSEMBLE SUMMARIZATION SYSTEM
# Three Fine-tuned Models: BART + LED + PEGASUS
# With InLegalBERT Mean Cosine Similarity Sentence Selection
# =========================================================

import os
import json
import torch
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration
)
from collections import Counter
import evaluate
import nltk

# Download NLTK data
print("📥 Downloading NLTK data...")
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION PARAMETERS
# =========================================================

# Model Paths - CHANGE THESE TO YOUR MODEL DIRECTORIES
MODEL_PATHS = {
    "BART": "BART",      # Path to your fine-tuned BART model
    "LED": "LED",        # Path to your fine-tuned LED model
    "PEGASUS": "PEGASUS" # Path to your fine-tuned PEGASUS model
}

# Data paths
TEST_JSON_PATH = "test.json"
OUTPUT_DIR = "./ensemble_results"

# Model-specific parameters
MODEL_CONFIG = {
    "BART": {
        "max_input": 1024,   # Maximum input tokens for BART
        "max_output": 256,   # Maximum output tokens
        "num_beams": 4       # Beam search width
    },
    "LED": {
        "max_input": 4096,   # LED handles longer inputs
        "max_output": 512,   # Longer outputs for LED
        "num_beams": 4
    },
    "PEGASUS": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    }
}

# Mean Cosine Similarity parameters
MCS_K = 60        # For BART and PEGASUS (number of sentences to select)
MCS_K_LED = 200   # LED can handle more sentences
MCS_THRESHOLD = 0.5  # Minimum similarity threshold (optional)

# Ensemble weights (adjust based on validation performance)
# These weights determine how much each model contributes to weighted ensemble
ENSEMBLE_WEIGHTS = {
    "BART": 0.35,
    "LED": 0.30,
    "PEGASUS": 0.35
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*70)
print("📋 CONFIGURATION SUMMARY")
print("="*70)
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Test data:         {TEST_JSON_PATH}")
print(f"Device:            {device}")
print(f"Selection method:  Mean Cosine Similarity")
print(f"Sentences (BART/PEGASUS): {MCS_K}")
print(f"Sentences (LED):   {MCS_K_LED}")
print(f"Ensemble weights:  BART={ENSEMBLE_WEIGHTS['BART']}, LED={ENSEMBLE_WEIGHTS['LED']}, PEGASUS={ENSEMBLE_WEIGHTS['PEGASUS']}")
print("="*70)

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 2: InLegalBERT EMBEDDING SYSTEM
# Sentence embedding generation for similarity computation
# =========================================================

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

# Assumes device is defined from Module 1
# device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# LOAD InLegalBERT FOR SENTENCE EMBEDDINGS
# =========================================================

def load_inlegalbert(device):
    """
    Load InLegalBERT model for generating sentence embeddings.
    
    Returns:
        tokenizer, model
    """
    print("\n📚 Loading InLegalBERT for embeddings...")
    legal_model_name = "law-ai/InLegalBERT"
    
    legal_tokenizer = AutoTokenizer.from_pretrained(legal_model_name)
    legal_model = AutoModel.from_pretrained(legal_model_name).to(device)
    legal_model.eval()
    
    print("✅ InLegalBERT loaded successfully")
    print(f"   Model: {legal_model_name}")
    print(f"   Parameters: {legal_model.num_parameters():,}")
    
    return legal_tokenizer, legal_model

# =========================================================
# EMBEDDING FUNCTION
# =========================================================

@torch.no_grad()
def embed_with_legalbert(texts, legal_tokenizer, legal_model, device, batch_size=16):
    """
    Compute sentence embeddings using InLegalBERT.
    
    Args:
        texts: List of text strings (sentences)
        legal_tokenizer: InLegalBERT tokenizer
        legal_model: InLegalBERT model
        device: torch device
        batch_size: Batch size for processing
        
    Returns:
        numpy array of embeddings [N, 768]
    """
    if len(texts) == 0:
        return np.array([])
    
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Tokenize batch
        encoded = legal_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        # Get model outputs
        outputs = legal_model(**encoded).last_hidden_state
        
        # Mean pooling with attention mask
        attention_mask = encoded["attention_mask"].unsqueeze(-1).float()
        pooled = (outputs * attention_mask).sum(1) / attention_mask.sum(1)
        
        all_embeddings.append(pooled.cpu().numpy())
    
    return np.vstack(all_embeddings)

# =========================================================
# TESTING FUNCTION
# =========================================================

def test_embedding_system(device):
    """Test the embedding system with sample sentences."""
    print("\n🧪 Testing embedding system...")
    
    # Load model
    tokenizer, model = load_inlegalbert(device)
    
    # Test sentences
    test_sentences = [
        "The appellant filed a petition under Article 32 of the Constitution.",
        "The respondent argued that the case should be dismissed.",
        "The court held that the petition was maintainable."
    ]
    
    # Generate embeddings
    embeddings = embed_with_legalbert(test_sentences, tokenizer, model, device)
    
    print(f"\n✅ Test successful!")
    print(f"   Input: {len(test_sentences)} sentences")
    print(f"   Output: {embeddings.shape} (shape)")
    print(f"   Embedding dimension: {embeddings.shape[1]}")
    
    return tokenizer, model

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    
    # Test the system
    tokenizer, model = test_embedding_system(device)
    
    print("\n✅ Module 2 ready to use!")
    print("\nUsage:")
    print("  from module2_embedding import load_inlegalbert, embed_with_legalbert")
    print("  tokenizer, model = load_inlegalbert(device)")
    print("  embeddings = embed_with_legalbert(sentences, tokenizer, model, device)")

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 3: MEAN COSINE SIMILARITY SENTENCE SELECTION
# Replaces MMR with Mean Cosine Similarity approach
# =========================================================

import numpy as np
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# MEAN COSINE SIMILARITY (MCS) ALGORITHM
# =========================================================

def select_sentences_mean_cosine_similarity(
    judgment, 
    embed_function,
    k=60,
    similarity_threshold=0.5
):
    """
    Select top-k sentences using Mean Cosine Similarity.
    
    Algorithm:
    1. Compute embeddings for all sentences
    2. Compute document centroid (mean of all embeddings)
    3. Calculate cosine similarity of each sentence to centroid
    4. Select top-k sentences with highest similarity
    5. Return sentences in original document order
    
    Args:
        judgment: Full legal document text
        embed_function: Function to generate embeddings
                       Should accept list of sentences, return numpy array
        k: Number of sentences to select
        similarity_threshold: Minimum similarity to include (optional filter)
        
    Returns:
        filtered_text: Selected sentences as single string
        selected_sentences: List of selected sentence strings
        selection_info: Dictionary with selection statistics
    """
    
    # Step 1: Tokenize into sentences
    sentences = sent_tokenize(judgment)
    n_original = len(sentences)
    
    # Handle edge case: fewer sentences than k
    if len(sentences) <= k:
        return judgment, sentences, {
            "method": "no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original,
            "compression_ratio": 1.0
        }
    
    # Step 2: Generate embeddings for all sentences
    embeddings = embed_function(sentences)
    
    # Step 3: Compute document centroid (mean of all embeddings)
    centroid = embeddings.mean(axis=0, keepdims=True)
    
    # Step 4: Calculate cosine similarity of each sentence to centroid
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    
    # Step 5: Apply threshold filter (optional)
    if similarity_threshold > 0:
        # Keep only sentences above threshold
        valid_indices = np.where(similarities >= similarity_threshold)[0]
        if len(valid_indices) < k:
            # If too few sentences pass threshold, use top-k anyway
            print(f"  ⚠️  Only {len(valid_indices)} sentences above threshold {similarity_threshold}")
            print(f"      Using top-{k} sentences regardless")
        else:
            # Filter to valid indices
            sentences = [sentences[i] for i in valid_indices]
            similarities = similarities[valid_indices]
            embeddings = embeddings[valid_indices]
    
    # Step 6: Select top-k sentences with highest similarity
    k_actual = min(k, len(sentences))
    top_k_indices = np.argsort(similarities)[-k_actual:]
    
    # Step 7: Sort indices to maintain document order
    selected_indices = sorted(top_k_indices)
    
    # Step 8: Extract selected sentences
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text = " ".join(selected_sentences)
    
    # Step 9: Compute statistics
    selection_info = {
        "method": "mean_cosine_similarity",
        "original_sents": n_original,
        "selected_sents": len(selected_sentences),
        "compression_ratio": len(selected_sentences) / n_original,
        "mean_similarity": float(np.mean(similarities[top_k_indices])),
        "min_similarity": float(np.min(similarities[top_k_indices])),
        "max_similarity": float(np.max(similarities[top_k_indices])),
        "threshold_used": similarity_threshold
    }
    
    return filtered_text, selected_sentences, selection_info

# =========================================================
# ALTERNATIVE: THRESHOLD-BASED SELECTION
# =========================================================

def select_sentences_by_threshold(
    judgment,
    embed_function,
    similarity_threshold=0.6,
    min_sentences=10,
    max_sentences=200
):
    """
    Select sentences purely by similarity threshold.
    
    Keeps all sentences above threshold (within min/max bounds).
    
    Args:
        judgment: Full legal document text
        embed_function: Function to generate embeddings
        similarity_threshold: Minimum similarity to document centroid
        min_sentences: Minimum number of sentences to keep
        max_sentences: Maximum number of sentences to keep
        
    Returns:
        filtered_text, selected_sentences, selection_info
    """
    sentences = sent_tokenize(judgment)
    n_original = len(sentences)
    
    if len(sentences) <= min_sentences:
        return judgment, sentences, {
            "method": "threshold_no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original
        }
    
    # Generate embeddings
    embeddings = embed_function(sentences)
    centroid = embeddings.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    
    # Find sentences above threshold
    above_threshold = similarities >= similarity_threshold
    n_above = np.sum(above_threshold)
    
    # Apply min/max bounds
    if n_above < min_sentences:
        # Take top min_sentences
        selected_indices = np.argsort(similarities)[-min_sentences:]
    elif n_above > max_sentences:
        # Take top max_sentences from above-threshold sentences
        candidate_indices = np.where(above_threshold)[0]
        candidate_sims = similarities[candidate_indices]
        top_candidates = np.argsort(candidate_sims)[-max_sentences:]
        selected_indices = candidate_indices[top_candidates]
    else:
        # Use all above-threshold sentences
        selected_indices = np.where(above_threshold)[0]
    
    # Sort to maintain order
    selected_indices = sorted(selected_indices)
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text = " ".join(selected_sentences)
    
    selection_info = {
        "method": "threshold_based",
        "original_sents": n_original,
        "selected_sents": len(selected_sentences),
        "compression_ratio": len(selected_sentences) / n_original,
        "threshold": similarity_threshold,
        "sentences_above_threshold": int(n_above),
        "mean_similarity": float(np.mean(similarities[selected_indices]))
    }
    
    return filtered_text, selected_sentences, selection_info

# =========================================================
# TESTING FUNCTION
# =========================================================

def test_selection_algorithm():
    """Test the selection algorithm with sample data."""
    print("\n🧪 Testing Mean Cosine Similarity selection...")
    
    # Sample judgment
    sample_judgment = """
    The Supreme Court of India delivered its judgment on 15th March 2023.
    The appellant challenged the constitutional validity of Section 377.
    The respondent argued that the provision was necessary for public morality.
    The Court examined the legislative history of the provision.
    International jurisprudence on similar provisions was also considered.
    The petitioner submitted detailed evidence of discrimination.
    The Attorney General defended the provision citing societal norms.
    The Court held that the provision violated Article 14 of the Constitution.
    The right to privacy was recognized as a fundamental right.
    The judgment was delivered by a five-judge constitutional bench.
    """
    
    # Mock embedding function (returns random embeddings for testing)
    def mock_embed_function(sentences):
        return np.random.randn(len(sentences), 768)
    
    # Test selection
    filtered_text, selected_sents, info = select_sentences_mean_cosine_similarity(
        sample_judgment,
        mock_embed_function,
        k=5
    )
    
    print(f"\n✅ Test successful!")
    print(f"   Original sentences: {info['original_sents']}")
    print(f"   Selected sentences: {info['selected_sents']}")
    print(f"   Compression ratio: {info['compression_ratio']:.2%}")
    print(f"   Mean similarity: {info['mean_similarity']:.4f}")
    print(f"\n   Selected sentences:")
    for i, sent in enumerate(selected_sents, 1):
        print(f"     {i}. {sent.strip()[:80]}...")
    
    return info

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    print("="*70)
    print("MODULE 3: MEAN COSINE SIMILARITY SENTENCE SELECTION")
    print("="*70)
    
    # Run test
    test_selection_algorithm()
    
    print("\n✅ Module 3 ready to use!")
    print("\nUsage:")
    print("  from module3_selection import select_sentences_mean_cosine_similarity")
    print("  filtered_text, sentences, info = select_sentences_mean_cosine_similarity(")
    print("      judgment, embed_function, k=60")
    print("  )")

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 4: MODEL LOADING
# Load BART, LED, and PEGASUS fine-tuned models
# =========================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration
)

# =========================================================
# MODEL LOADING FUNCTIONS
# =========================================================

def load_bart_model(model_path, device):
    """
    Load fine-tuned BART model.
    
    Args:
        model_path: Path to BART model directory
        device: torch device
        
    Returns:
        tokenizer, model
    """
    print("\n  [1/3] Loading BART...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()
    print(f"  ✅ BART loaded - {model.num_parameters():,} parameters")
    return tokenizer, model

def load_led_model(model_path, device):
    """
    Load fine-tuned LED model.
    
    Args:
        model_path: Path to LED model directory
        device: torch device
        
    Returns:
        tokenizer, model
    """
    print("\n  [2/3] Loading LED...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = LEDForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()
    print(f"  ✅ LED loaded - {model.num_parameters():,} parameters")
    return tokenizer, model

def load_pegasus_model(model_path, device):
    """
    Load fine-tuned PEGASUS model.
    
    Args:
        model_path: Path to PEGASUS model directory
        device: torch device
        
    Returns:
        tokenizer, model
    """
    print("\n  [3/3] Loading PEGASUS...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = PegasusForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()
    print(f"  ✅ PEGASUS loaded - {model.num_parameters():,} parameters")
    return tokenizer, model

# =========================================================
# LOAD ALL MODELS
# =========================================================

def load_all_models(model_paths, device):
    """
    Load all three models at once.
    
    Args:
        model_paths: Dictionary with keys "BART", "LED", "PEGASUS"
        device: torch device
        
    Returns:
        models: Dictionary of loaded models
        tokenizers: Dictionary of loaded tokenizers
    """
    print("\n📚 Loading all three fine-tuned models...")
    print("="*70)
    
    models = {}
    tokenizers = {}
    
    # Load BART
    tokenizers["BART"], models["BART"] = load_bart_model(
        model_paths["BART"], 
        device
    )
    
    # Load LED
    tokenizers["LED"], models["LED"] = load_led_model(
        model_paths["LED"], 
        device
    )
    
    # Load PEGASUS
    tokenizers["PEGASUS"], models["PEGASUS"] = load_pegasus_model(
        model_paths["PEGASUS"], 
        device
    )
    
    print("\n✅ All models loaded successfully!")
    print("="*70)
    
    return models, tokenizers

# =========================================================
# MODEL INFO FUNCTION
# =========================================================

def print_model_info(models):
    """Print information about loaded models."""
    print("\n📊 Model Information:")
    print("-"*70)
    
    for model_name, model in models.items():
        print(f"\n{model_name}:")
        print(f"  Parameters: {model.num_parameters():,}")
        print(f"  Type: {type(model).__name__}")
        print(f"  Device: {next(model.parameters()).device}")
    
    print("-"*70)

# =========================================================
# TESTING FUNCTION
# =========================================================

def test_model_loading(model_paths, device):
    """Test loading all models."""
    print("🧪 Testing model loading...")
    
    try:
        models, tokenizers = load_all_models(model_paths, device)
        print_model_info(models)
        
        # Test tokenization
        print("\n🧪 Testing tokenization...")
        test_text = "The Supreme Court delivered its judgment."
        
        for model_name in ["BART", "LED", "PEGASUS"]:
            tokens = tokenizers[model_name](test_text, return_tensors="pt")
            print(f"  {model_name}: {tokens['input_ids'].shape[1]} tokens")
        
        print("\n✅ All tests passed!")
        return models, tokenizers
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    # Configuration
    MODEL_PATHS = {
        "BART": "BART",
        "LED": "LED",
        "PEGASUS": "PEGASUS"
    }
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    
    # Load models
    models, tokenizers = test_model_loading(MODEL_PATHS, device)
    
    if models and tokenizers:
        print("\n✅ Module 4 ready to use!")
        print("\nUsage:")
        print("  from module4_models import load_all_models")
        print("  models, tokenizers = load_all_models(model_paths, device)")

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 5: SUMMARY GENERATION
# Generate summaries using individual models
# =========================================================

import torch

# =========================================================
# SINGLE MODEL SUMMARY GENERATION
# =========================================================

def generate_summary_single_model(
    model_name,
    judgment,
    models,
    tokenizers,
    model_config,
    selection_function,
    k_value
):
    """
    Generate summary using a single model.
    
    Args:
        model_name: Name of model ("BART", "LED", or "PEGASUS")
        judgment: Full legal document text
        models: Dictionary of loaded models
        tokenizers: Dictionary of loaded tokenizers
        model_config: Dictionary with model configurations
        selection_function: Function to select sentences
        k_value: Number of sentences to select
        
    Returns:
        summary: Generated summary text
        selection_info: Information about sentence selection
    """
    model = models[model_name]
    tokenizer = tokenizers[model_name]
    config = model_config[model_name]
    
    # Step 1: Select sentences using Mean Cosine Similarity
    filtered_text, selected_sentences, selection_info = selection_function(
        judgment,
        k=k_value
    )
    
    # Step 2: Tokenize the filtered text
    inputs = tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input"]
    ).to(model.device)
    
    # Step 3: Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=config["max_output"],
            num_beams=config["num_beams"],
            early_stopping=True,
            no_repeat_ngram_size=3  # Avoid repetition
        )
    
    # Step 4: Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return summary, selection_info

# =========================================================
# BATCH SUMMARY GENERATION
# =========================================================

def generate_summaries_for_all_models(
    judgment,
    reference,
    models,
    tokenizers,
    model_config,
    selection_function,
    mcs_k,
    mcs_k_led
):
    """
    Generate summaries using all three models.
    
    Args:
        judgment: Full legal document text
        reference: Reference summary (for evaluation)
        models: Dictionary of loaded models
        tokenizers: Dictionary of loaded tokenizers
        model_config: Dictionary with model configurations
        selection_function: Function to select sentences
        mcs_k: Number of sentences for BART/PEGASUS
        mcs_k_led: Number of sentences for LED
        
    Returns:
        summaries_dict: Dictionary with summaries from each model
        selection_info_dict: Dictionary with selection info from each model
    """
    summaries_dict = {}
    selection_info_dict = {}
    
    # Generate summary with each model
    for model_name in ["BART", "LED", "PEGASUS"]:
        # Use appropriate k value
        k_value = mcs_k_led if model_name == "LED" else mcs_k
        
        # Generate summary
        summary, selection_info = generate_summary_single_model(
            model_name=model_name,
            judgment=judgment,
            models=models,
            tokenizers=tokenizers,
            model_config=model_config,
            selection_function=selection_function,
            k_value=k_value
        )
        
        summaries_dict[model_name] = summary
        selection_info_dict[model_name] = selection_info
    
    return summaries_dict, selection_info_dict

# =========================================================
# DETAILED SUMMARY GENERATION (WITH STATS)
# =========================================================

def generate_summary_with_stats(
    model_name,
    judgment,
    reference,
    models,
    tokenizers,
    model_config,
    selection_function,
    k_value
):
    """
    Generate summary and return detailed statistics.
    
    Returns:
        Dictionary with:
        - summary: Generated summary
        - input_length: Number of input tokens
        - output_length: Number of output tokens
        - selection_info: Sentence selection information
        - original_doc_length: Original document length
        - filtered_doc_length: Filtered document length
    """
    model = models[model_name]
    tokenizer = tokenizers[model_name]
    config = model_config[model_name]
    
    # Select sentences
    filtered_text, selected_sentences, selection_info = selection_function(
        judgment,
        k=k_value
    )
    
    # Tokenize
    inputs = tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input"]
    ).to(model.device)
    
    # Generate
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=config["max_output"],
            num_beams=config["num_beams"],
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    # Compute statistics
    stats = {
        "summary": summary,
        "input_length": inputs["input_ids"].shape[1],
        "output_length": len(tokenizer.encode(summary)),
        "selection_info": selection_info,
        "original_doc_length": len(judgment.split()),
        "filtered_doc_length": len(filtered_text.split()),
        "compression_ratio_selection": selection_info["compression_ratio"],
        "compression_ratio_overall": len(summary.split()) / len(judgment.split())
    }
    
    return stats

# =========================================================
# TESTING FUNCTION
# =========================================================

def test_summary_generation():
    """Test summary generation with mock data."""
    print("\n🧪 Testing summary generation...")
    
    # Mock data
    sample_judgment = "The Supreme Court delivered its judgment. " * 50
    sample_reference = "The Court ruled in favor of the appellant."
    
    # Mock selection function
    def mock_selection_function(judgment, k=60):
        from nltk.tokenize import sent_tokenize
        sents = sent_tokenize(judgment)
        selected = sents[:k]
        return " ".join(selected), selected, {
            "method": "mock",
            "original_sents": len(sents),
            "selected_sents": len(selected),
            "compression_ratio": len(selected) / len(sents)
        }
    
    print("  Mock selection function created")
    print(f"  Sample judgment: {len(sample_judgment.split())} words")
    
    print("\n✅ Test structure verified!")
    print("  (Full test requires loaded models)")
    
    return True

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    print("="*70)
    print("MODULE 5: SUMMARY GENERATION")
    print("="*70)
    
    # Run test
    test_summary_generation()
    
    print("\n✅ Module 5 ready to use!")
    print("\nUsage:")
    print("  from module5_generation import generate_summaries_for_all_models")
    print("  summaries, info = generate_summaries_for_all_models(")
    print("      judgment, reference, models, tokenizers, ...")
    print("  )")

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 6: ENSEMBLE FUSION STRATEGIES
# Different methods to combine multiple model outputs
# =========================================================

import numpy as np
from nltk.tokenize import sent_tokenize
from collections import Counter
import evaluate

# =========================================================
# STRATEGY 1: VOTING-BASED ENSEMBLE
# =========================================================

def ensemble_voting(summaries_dict, threshold=2):
    """
    Voting-based ensemble: Select sentences appearing in ≥threshold models.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        threshold: Minimum number of models that must include a sentence
        
    Returns:
        combined_summary: Ensemble summary
    """
    all_sentences = []
    
    # Collect all sentences from all models
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        all_sentences.extend(sents)
    
    # Count sentence occurrences (normalized for matching)
    sentence_counts = Counter()
    sentence_originals = {}  # Keep original capitalization
    
    for sent in all_sentences:
        # Normalize: lowercase and strip for counting
        normalized = sent.lower().strip()
        sentence_counts[normalized] += 1
        
        # Store original if not stored yet
        if normalized not in sentence_originals:
            sentence_originals[normalized] = sent
    
    # Select sentences appearing in at least 'threshold' models
    selected = [
        sentence_originals[sent] 
        for sent, count in sentence_counts.items() 
        if count >= threshold
    ]
    
    # Fallback: if too few sentences, use most common ones
    if len(selected) < 3:
        selected = [
            sentence_originals[sent] 
            for sent, _ in sentence_counts.most_common(10)
        ]
    
    return " ".join(selected)

# =========================================================
# STRATEGY 2: WEIGHTED CONCATENATION
# =========================================================

def ensemble_weighted_concat(summaries_dict, weights):
    """
    Weighted concatenation: Combine summaries with quality-based weights.
    
    Takes a proportion of sentences from each model based on its weight.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        weights: Dictionary {model_name: weight} (should sum to 1.0)
        
    Returns:
        combined_summary: Ensemble summary
    """
    weighted_parts = []
    
    for model_name, summary in summaries_dict.items():
        weight = weights.get(model_name, 0.33)  # Default equal weight
        sents = sent_tokenize(summary)
        
        # Number of sentences to take based on weight
        n_sents = max(1, int(len(sents) * weight))
        weighted_parts.extend(sents[:n_sents])
    
    # Remove duplicates while preserving order
    seen = set()
    unique_sents = []
    
    for sent in weighted_parts:
        normalized = sent.lower().strip()
        if normalized not in seen:
            seen.add(normalized)
            unique_sents.append(sent)
    
    return " ".join(unique_sents)

# =========================================================
# STRATEGY 3: BEST MODEL SELECTION
# =========================================================

def ensemble_best_model(summaries_dict, reference, rouge_metric=None):
    """
    Select best single summary based on ROUGE-2 score with reference.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        reference: Reference summary for comparison
        rouge_metric: evaluate.load("rouge") instance
        
    Returns:
        best_summary: Summary from best-performing model
    """
    if rouge_metric is None:
        rouge_metric = evaluate.load("rouge")
    
    best_score = -1
    best_summary = ""
    best_model = ""
    
    for model_name, summary in summaries_dict.items():
        # Compute ROUGE-2 score
        score = rouge_metric.compute(
            predictions=[summary],
            references=[reference],
            use_stemmer=True
        )["rouge2"]
        
        if score > best_score:
            best_score = score
            best_summary = summary
            best_model = model_name
    
    return best_summary

# =========================================================
# STRATEGY 4: SENTENCE RANKING
# =========================================================

def ensemble_sentence_ranking(summaries_dict, top_n=15):
    """
    Rank-based fusion: Score each sentence by appearance position across models.
    
    Sentences appearing earlier in multiple models get higher scores.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        top_n: Number of top sentences to include
        
    Returns:
        combined_summary: Ensemble summary
    """
    sentence_positions = {}
    sentence_originals = {}
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        
        for pos, sent in enumerate(sents):
            normalized = sent.lower().strip()
            
            # Store original
            if normalized not in sentence_originals:
                sentence_originals[normalized] = sent
            
            # Record position (lower position = more important)
            if normalized not in sentence_positions:
                sentence_positions[normalized] = []
            sentence_positions[normalized].append(pos)
    
    # Compute score: average position (lower is better)
    sentence_scores = {
        sent: np.mean(positions) 
        for sent, positions in sentence_positions.items()
    }
    
    # Sort by score (ascending - lower position = better)
    ranked = sorted(sentence_scores.items(), key=lambda x: x[1])
    
    # Take top N sentences
    top_sentences = [sentence_originals[sent] for sent, _ in ranked[:top_n]]
    
    return " ".join(top_sentences)

# =========================================================
# STRATEGY 5: MEAN POOLING (ADVANCED)
# =========================================================

def ensemble_mean_pooling(summaries_dict, embed_function, top_n=15):
    """
    Advanced: Use embeddings to select most representative sentences.
    
    Combines sentences from all models and selects those closest to
    the mean embedding of all summary sentences.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        embed_function: Function to generate sentence embeddings
        top_n: Number of sentences to select
        
    Returns:
        combined_summary: Ensemble summary
    """
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Collect all unique sentences
    all_sentences = []
    sentence_originals = {}
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for sent in sents:
            normalized = sent.lower().strip()
            if normalized not in sentence_originals:
                sentence_originals[normalized] = sent
                all_sentences.append(sent)
    
    if len(all_sentences) == 0:
        return ""
    
    # Generate embeddings
    embeddings = embed_function(all_sentences)
    
    # Compute mean embedding
    mean_embedding = embeddings.mean(axis=0, keepdims=True)
    
    # Calculate similarity to mean
    similarities = cosine_similarity(embeddings, mean_embedding).squeeze()
    
    # Select top-N most similar
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    top_sentences = [all_sentences[i] for i in top_indices]
    
    return " ".join(top_sentences)

# =========================================================
# APPLY ALL ENSEMBLE STRATEGIES
# =========================================================

def apply_all_ensemble_strategies(
    summaries_dict,
    reference,
    weights,
    embed_function=None,
    rouge_metric=None
):
    """
    Apply all ensemble strategies and return results.
    
    Args:
        summaries_dict: Dictionary {model_name: summary_text}
        reference: Reference summary
        weights: Dictionary {model_name: weight}
        embed_function: Optional embedding function for mean pooling
        rouge_metric: Optional rouge metric instance
        
    Returns:
        ensemble_results: Dictionary {strategy_name: summary}
    """
    results = {}
    
    # Strategy 1: Voting
    results["voting"] = ensemble_voting(summaries_dict, threshold=2)
    
    # Strategy 2: Weighted
    results["weighted"] = ensemble_weighted_concat(summaries_dict, weights)
    
    # Strategy 3: Best Model
    results["best_model"] = ensemble_best_model(
        summaries_dict, reference, rouge_metric
    )
    
    # Strategy 4: Ranking
    results["ranking"] = ensemble_sentence_ranking(summaries_dict, top_n=15)
    
    # Strategy 5: Mean Pooling (if embedding function provided)
    if embed_function is not None:
        results["mean_pooling"] = ensemble_mean_pooling(
            summaries_dict, embed_function, top_n=15
        )
    
    return results

# =========================================================
# TESTING FUNCTION
# =========================================================

def test_ensemble_strategies():
    """Test ensemble strategies with mock data."""
    print("\n🧪 Testing ensemble strategies...")
    
    # Mock summaries from three models
    summaries_dict = {
        "BART": "The Court ruled in favor. The appellant was granted relief. Justice was served.",
        "LED": "The Court ruled in favor. The petition was allowed. The case is closed.",
        "PEGASUS": "The appellant was granted relief. The petition was allowed. Justice prevailed."
    }
    
    reference = "The Court ruled in favor of the appellant."
    
    weights = {"BART": 0.35, "LED": 0.30, "PEGASUS": 0.35}
    
    # Test each strategy
    print("\n1. Voting Ensemble:")
    result = ensemble_voting(summaries_dict)
    print(f"   {result[:100]}...")
    
    print("\n2. Weighted Ensemble:")
    result = ensemble_weighted_concat(summaries_dict, weights)
    print(f"   {result[:100]}...")
    
    print("\n3. Best Model:")
    result = ensemble_best_model(summaries_dict, reference)
    print(f"   {result[:100]}...")
    
    print("\n4. Sentence Ranking:")
    result = ensemble_sentence_ranking(summaries_dict)
    print(f"   {result[:100]}...")
    
    print("\n✅ All strategies tested!")
    
    return True

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    print("="*70)
    print("MODULE 6: ENSEMBLE FUSION STRATEGIES")
    print("="*70)
    
    # Run test
    test_ensemble_strategies()
    
    print("\n✅ Module 6 ready to use!")
    print("\nAvailable strategies:")
    print("  1. ensemble_voting() - Select common sentences")
    print("  2. ensemble_weighted_concat() - Weighted combination")
    print("  3. ensemble_best_model() - Pick best performer")
    print("  4. ensemble_sentence_ranking() - Rank by position")
    print("  5. ensemble_mean_pooling() - Embedding-based selection")

In [ ]:
#!/usr/bin/env python3
# =========================================================
# MODULE 7: EVALUATION AND MAIN PIPELINE
# Complete pipeline with evaluation
# =========================================================

import json
import numpy as np
from tqdm import tqdm
import evaluate

# =========================================================
# EVALUATION METRICS
# =========================================================

def load_evaluation_metrics(device):
    """
    Load ROUGE and BERTScore metrics.
    
    Returns:
        rouge_metric, bertscore_metric
    """
    print("\n📊 Loading evaluation metrics...")
    
    rouge_metric = evaluate.load("rouge")
    bertscore_metric = evaluate.load("bertscore")
    
    print("✅ Metrics loaded: ROUGE, BERTScore")
    
    return rouge_metric, bertscore_metric

def compute_metrics(predictions, references, rouge_metric, bertscore_metric, device):
    """
    Compute ROUGE and BERTScore metrics.
    
    Args:
        predictions: List of generated summaries
        references: List of reference summaries
        rouge_metric: ROUGE metric instance
        bertscore_metric: BERTScore metric instance
        device: torch device
        
    Returns:
        Dictionary with all metrics
    """
    # Compute ROUGE scores
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    # Compute BERTScore
    bert_scores = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        model_type="roberta-large",
        lang="en",
        device=device,
        batch_size=8
    )
    
    return {
        "rouge1": float(rouge_scores["rouge1"]),
        "rouge2": float(rouge_scores["rouge2"]),
        "rougeL": float(rouge_scores["rougeL"]),
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall": float(np.mean(bert_scores["recall"])),
        "bertscore_f1": float(np.mean(bert_scores["f1"])),
    }

# =========================================================
# PROCESS ALL TEST SAMPLES
# =========================================================

def process_all_test_samples(
    test_data,
    generate_summaries_function,
    apply_ensemble_function,
    ensemble_weights
):
    """
    Process all test samples through all models and ensemble strategies.
    
    Args:
        test_data: List of test samples
        generate_summaries_function: Function to generate summaries
        apply_ensemble_function: Function to apply ensemble strategies
        ensemble_weights: Dictionary of model weights
        
    Returns:
        all_model_summaries: Dictionary of summaries per model
        ensemble_summaries: Dictionary of summaries per strategy
        references: List of reference summaries
    """
    print("\n" + "="*70)
    print("🚀 GENERATING SUMMARIES WITH ALL THREE MODELS")
    print("="*70)
    
    all_model_summaries = {
        "BART": [],
        "LED": [],
        "PEGASUS": []
    }
    
    ensemble_summaries = {
        "voting": [],
        "weighted": [],
        "best_model": [],
        "ranking": []
    }
    
    references = []
    
    for idx, item in enumerate(tqdm(test_data, desc="Processing samples")):
        judgment = item["judgment"]
        reference = item["reference_summary"]
        references.append(reference)
        
        # Generate summary with each model
        summaries_dict, _ = generate_summaries_function(judgment, reference)
        
        # Store individual model summaries
        for model_name in ["BART", "LED", "PEGASUS"]:
            all_model_summaries[model_name].append(summaries_dict[model_name])
        
        # Apply ensemble strategies
        ensemble_results = apply_ensemble_function(
            summaries_dict, 
            reference, 
            ensemble_weights
        )
        
        for strategy in ["voting", "weighted", "best_model", "ranking"]:
            ensemble_summaries[strategy].append(ensemble_results[strategy])
    
    print("\n✅ All summaries generated!")
    
    return all_model_summaries, ensemble_summaries, references

# =========================================================
# EVALUATE ALL APPROACHES
# =========================================================

def evaluate_all_approaches(
    all_model_summaries,
    ensemble_summaries,
    references,
    rouge_metric,
    bertscore_metric,
    device
):
    """
    Evaluate all models and ensemble strategies.
    
    Returns:
        results: Dictionary with metrics for each approach
    """
    print("\n" + "="*70)
    print("📊 EVALUATING ALL APPROACHES")
    print("="*70)
    
    results = {}
    
    # Evaluate individual models
    print("\nEvaluating individual models...")
    for model_name in ["BART", "LED", "PEGASUS"]:
        print(f"  Evaluating {model_name}...")
        metrics = compute_metrics(
            all_model_summaries[model_name],
            references,
            rouge_metric,
            bertscore_metric,
            device
        )
        results[model_name] = metrics
        
        print(f"  ✅ {model_name} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")
    
    # Evaluate ensemble strategies
    print("\nEvaluating ensemble strategies...")
    for strategy in ["voting", "weighted", "best_model", "ranking"]:
        print(f"  Evaluating Ensemble-{strategy}...")
        metrics = compute_metrics(
            ensemble_summaries[strategy],
            references,
            rouge_metric,
            bertscore_metric,
            device
        )
        results[f"ensemble_{strategy}"] = metrics
        
        print(f"  ✅ Ensemble-{strategy} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")
    
    return results

# =========================================================
# PRINT COMPARISON TABLE
# =========================================================

def print_comparison_table(results):
    """Print comprehensive results comparison table."""
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE RESULTS COMPARISON")
    print("="*70)
    
    print(f"\n{'Approach':<20} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BERTScore F1':<12}")
    print("-" * 70)
    
    for approach_name, metrics in results.items():
        print(f"{approach_name:<20} {metrics['rouge1']:<10.4f} {metrics['rouge2']:<10.4f} "
              f"{metrics['rougeL']:<10.4f} {metrics['bertscore_f1']:<12.4f}")
    
    # Find best approach
    best_approach = max(results.items(), key=lambda x: x[1]['rouge2'])
    
    print("\n" + "="*70)
    print(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
    print(f"   ROUGE-2: {best_approach[1]['rouge2']:.4f}")
    print(f"   BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
    print("="*70)
    
    return best_approach

# =========================================================
# SAVE RESULTS
# =========================================================

def save_all_results(
    output_dir,
    test_data,
    all_model_summaries,
    ensemble_summaries,
    references,
    results,
    best_approach,
    ensemble_weights,
    mcs_k,
    mcs_k_led
):
    """Save all results to files."""
    import os
    
    print("\n💾 Saving results...")
    
    # Save individual model summaries
    for model_name in ["BART", "LED", "PEGASUS"]:
        output_path = os.path.join(output_dir, f"{model_name.lower()}_summaries.json")
        with open(output_path, 'w', encoding='utf8') as f:
            json.dump([
                {
                    "id": item.get("id", idx),
                    "generated_summary": summary,
                    "reference_summary": ref
                }
                for idx, (item, summary, ref) in enumerate(
                    zip(test_data, all_model_summaries[model_name], references)
                )
            ], f, indent=2, ensure_ascii=False)
        print(f"  ✅ Saved {output_path}")
    
    # Save ensemble summaries
    for strategy in ["voting", "weighted", "best_model", "ranking"]:
        output_path = os.path.join(output_dir, f"ensemble_{strategy}_summaries.json")
        with open(output_path, 'w', encoding='utf8') as f:
            json.dump([
                {
                    "id": item.get("id", idx),
                    "generated_summary": summary,
                    "reference_summary": ref
                }
                for idx, (item, summary, ref) in enumerate(
                    zip(test_data, ensemble_summaries[strategy], references)
                )
            ], f, indent=2, ensure_ascii=False)
        print(f"  ✅ Saved {output_path}")
    
    # Save complete results
    complete_results = {
        "experiment": "Ensemble Summarization - BART + LED + PEGASUS (Mean Cosine Similarity)",
        "test_samples": len(test_data),
        "selection_method": "Mean Cosine Similarity",
        "mcs_k": mcs_k,
        "mcs_k_led": mcs_k_led,
        "ensemble_weights": ensemble_weights,
        "results": results,
        "best_approach": {
            "name": best_approach[0],
            "metrics": best_approach[1]
        }
    }
    
    results_path = os.path.join(output_dir, "ensemble_complete_results.json")
    with open(results_path, 'w', encoding='utf8') as f:
        json.dump(complete_results, f, indent=2, ensure_ascii=False)
    
    print(f"\n  ✅ Saved {results_path}")
    
    return complete_results

# =========================================================
# GENERATE REPORT
# =========================================================

def generate_report(
    output_dir,
    test_data,
    results,
    best_approach,
    ensemble_weights,
    mcs_k,
    mcs_k_led
):
    """Generate comprehensive text report."""
    import os
    
    report_lines = []
    report_lines.append("="*70)
    report_lines.append("ENSEMBLE SUMMARIZATION EXPERIMENT")
    report_lines.append("BART + LED + PEGASUS Fine-tuned Models")
    report_lines.append("Sentence Selection: Mean Cosine Similarity")
    report_lines.append("="*70)
    report_lines.append("")
    report_lines.append(f"Test Samples: {len(test_data)}")
    report_lines.append(f"Selection Method: Mean Cosine Similarity")
    report_lines.append(f"MCS Configuration: k={mcs_k} (BART/PEGASUS), k={mcs_k_led} (LED)")
    report_lines.append(f"Ensemble Weights: BART={ensemble_weights['BART']}, LED={ensemble_weights['LED']}, PEGASUS={ensemble_weights['PEGASUS']}")
    report_lines.append("")
    report_lines.append("-"*70)
    report_lines.append("INDIVIDUAL MODEL RESULTS")
    report_lines.append("-"*70)
    
    for model_name in ["BART", "LED", "PEGASUS"]:
        m = results[model_name]
        report_lines.append(f"\n{model_name}:")
        report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
        report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
        report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
        report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")
    
    report_lines.append("")
    report_lines.append("-"*70)
    report_lines.append("ENSEMBLE RESULTS")
    report_lines.append("-"*70)
    
    for strategy in ["voting", "weighted", "best_model", "ranking"]:
        m = results[f"ensemble_{strategy}"]
        report_lines.append(f"\nEnsemble-{strategy.upper()}:")
        report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
        report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
        report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
        report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")
    
    report_lines.append("")
    report_lines.append("="*70)
    report_lines.append(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
    report_lines.append("="*70)
    report_lines.append(f"ROUGE-2:      {best_approach[1]['rouge2']:.4f}")
    report_lines.append(f"BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
    report_lines.append("="*70)
    
    report_text = "\n".join(report_lines)
    print("\n" + report_text)
    
    report_path = os.path.join(output_dir, "ensemble_report.txt")
    with open(report_path, 'w', encoding='utf8') as f:
        f.write(report_text)
    
    print(f"\n💾 Report saved to: {report_path}")
    
    return report_path

# =========================================================
# USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    print("="*70)
    print("MODULE 7: EVALUATION AND MAIN PIPELINE")
    print("="*70)
    
    print("\n✅ Module 7 ready to use!")
    print("\nThis module provides:")
    print("  - load_evaluation_metrics()")
    print("  - compute_metrics()")
    print("  - process_all_test_samples()")
    print("  - evaluate_all_approaches()")
    print("  - print_comparison_table()")
    print("  - save_all_results()")
    print("  - generate_report()")

In [1]:
#!/usr/bin/env python3
# =========================================================
# COMPLETE ENSEMBLE SUMMARIZATION PIPELINE
# Mean Cosine Similarity + BART + LED + PEGASUS
# =========================================================

import os
import json
import torch
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration
)
from collections import Counter
import evaluate
import nltk

# Download NLTK data
print("📥 Downloading NLTK data...")
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION
# =========================================================

# Model Paths
MODEL_PATHS = {
    "BART": "BART",
    "LED": "LED",
    "PEGASUS": "PEGASUS"
}

# Data paths
TEST_JSON_PATH = "test.json"
OUTPUT_DIR = "./ensemble_results_mcs"

# Model-specific parameters
MODEL_CONFIG = {
    "BART": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    },
    "LED": {
        "max_input": 4096,
        "max_output": 512,
        "num_beams": 4
    },
    "PEGASUS": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    }
}

# Mean Cosine Similarity parameters
MCS_K = 60        # For BART and PEGASUS
MCS_K_LED = 200   # LED can handle more sentences

# Ensemble weights
ENSEMBLE_WEIGHTS = {
    "BART": 0.35,
    "LED": 0.30,
    "PEGASUS": 0.35
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*70)
print("📋 CONFIGURATION")
print("="*70)
print(f"Selection Method: Mean Cosine Similarity")
print(f"Sentences (BART/PEGASUS): {MCS_K}")
print(f"Sentences (LED): {MCS_K_LED}")
print(f"Output directory: {OUTPUT_DIR}")
print("="*70)

# =========================================================
# LOAD InLegalBERT FOR SENTENCE EMBEDDINGS
# =========================================================

print("\n📚 Loading InLegalBERT...")
legal_model_name = "law-ai/InLegalBERT"
legal_tokenizer = AutoTokenizer.from_pretrained(legal_model_name)
legal_model = AutoModel.from_pretrained(legal_model_name).to(device)
legal_model.eval()
print("✅ InLegalBERT loaded successfully")

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    """Compute sentence embeddings using InLegalBERT."""
    if len(texts) == 0:
        return np.array([])
    
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        enc = legal_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        out = legal_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1)
        
        embs.append(pooled.cpu().numpy())
    
    return np.vstack(embs)

# =========================================================
# MEAN COSINE SIMILARITY SENTENCE SELECTION
# =========================================================

def select_sentences_mean_cosine_similarity(judgment, k=60):
    """
    Select top-k sentences using Mean Cosine Similarity.
    
    Algorithm:
    1. Compute embeddings for all sentences
    2. Compute document centroid (mean of all embeddings)
    3. Calculate cosine similarity to centroid
    4. Select top-k sentences with highest similarity
    """
    sentences = sent_tokenize(judgment)
    n_original = len(sentences)
    
    if len(sentences) <= k:
        return judgment, sentences, {
            "method": "no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original
        }
    
    # Generate embeddings
    embeddings = embed_with_legalbert(sentences)
    
    # Compute document centroid
    centroid = embeddings.mean(axis=0, keepdims=True)
    
    # Calculate cosine similarity to centroid
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    
    # Select top-k sentences
    top_k_indices = np.argsort(similarities)[-k:]
    
    # Sort to maintain document order
    selected_indices = sorted(top_k_indices)
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text = " ".join(selected_sentences)
    
    selection_info = {
        "method": "mean_cosine_similarity",
        "original_sents": n_original,
        "selected_sents": len(selected_indices),
        "compression_ratio": len(selected_indices) / n_original,
        "mean_similarity": float(np.mean(similarities[top_k_indices]))
    }
    
    return filtered_text, selected_sentences, selection_info

print("✅ Mean Cosine Similarity selection ready")

# =========================================================
# LOAD ALL THREE MODELS
# =========================================================

print("\n📚 Loading all three fine-tuned models...")
models = {}
tokenizers = {}

# Load BART
print("\n  [1/3] Loading BART...")
tokenizers["BART"] = AutoTokenizer.from_pretrained(MODEL_PATHS["BART"])
models["BART"] = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATHS["BART"]).to(device)
models["BART"].eval()
print(f"  ✅ BART loaded - {models['BART'].num_parameters():,} parameters")

# Load LED
print("\n  [2/3] Loading LED...")
tokenizers["LED"] = AutoTokenizer.from_pretrained(MODEL_PATHS["LED"])
models["LED"] = LEDForConditionalGeneration.from_pretrained(MODEL_PATHS["LED"]).to(device)
models["LED"].eval()
print(f"  ✅ LED loaded - {models['LED'].num_parameters():,} parameters")

# Load PEGASUS
print("\n  [3/3] Loading PEGASUS...")
tokenizers["PEGASUS"] = AutoTokenizer.from_pretrained(MODEL_PATHS["PEGASUS"])
models["PEGASUS"] = PegasusForConditionalGeneration.from_pretrained(MODEL_PATHS["PEGASUS"]).to(device)
models["PEGASUS"].eval()
print(f"  ✅ PEGASUS loaded - {models['PEGASUS'].num_parameters():,} parameters")

print("\n✅ All models loaded successfully!")

# =========================================================
# LOAD TEST DATA
# =========================================================

print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")

with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
    test_data = json.load(f)

print(f"✅ Loaded {len(test_data)} test samples")

# =========================================================
# EVALUATION METRICS
# =========================================================

print("\n📊 Loading evaluation metrics...")

rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ Metrics loaded: ROUGE, BERTScore")

def compute_metrics(predictions, references):
    """Compute ROUGE and BERTScore metrics."""
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    bert_scores = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        model_type="roberta-large",
        lang="en",
        device=device,
        batch_size=8
    )
    
    return {
        "rouge1": float(rouge_scores["rouge1"]),
        "rouge2": float(rouge_scores["rouge2"]),
        "rougeL": float(rouge_scores["rougeL"]),
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall": float(np.mean(bert_scores["recall"])),
        "bertscore_f1": float(np.mean(bert_scores["f1"])),
    }

# =========================================================
# SINGLE MODEL SUMMARY GENERATION
# =========================================================

def generate_summary(model_name, judgment, reference):
    """Generate summary using a single model."""
    model = models[model_name]
    tokenizer = tokenizers[model_name]
    config = MODEL_CONFIG[model_name]
    
    # Use appropriate k value
    k_value = MCS_K_LED if model_name == "LED" else MCS_K
    
    # Select sentences using Mean Cosine Similarity
    filtered_text, _, selection_info = select_sentences_mean_cosine_similarity(
        judgment, 
        k=k_value
    )
    
    # Tokenize
    inputs = tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input"]
    ).to(device)
    
    # Generate
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=config["max_output"],
            num_beams=config["num_beams"],
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return summary, selection_info

# =========================================================
# ENSEMBLE FUSION STRATEGIES
# =========================================================

def ensemble_voting(summaries_dict):
    """Voting-based ensemble."""
    all_sentences = []
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        all_sentences.extend(sents)
    
    sentence_counts = Counter()
    for sent in all_sentences:
        normalized = sent.lower().strip()
        sentence_counts[normalized] += 1
    
    threshold = 2
    selected = [sent for sent, count in sentence_counts.items() if count >= threshold]
    
    if len(selected) < 3:
        selected = [sent for sent, _ in sentence_counts.most_common(10)]
    
    return " ".join(selected)

def ensemble_weighted_concat(summaries_dict, weights):
    """Weighted concatenation."""
    weighted_parts = []
    
    for model_name, summary in summaries_dict.items():
        weight = weights[model_name]
        sents = sent_tokenize(summary)
        n_sents = max(1, int(len(sents) * weight))
        weighted_parts.extend(sents[:n_sents])
    
    seen = set()
    unique_sents = []
    for sent in weighted_parts:
        normalized = sent.lower().strip()
        if normalized not in seen:
            seen.add(normalized)
            unique_sents.append(sent)
    
    return " ".join(unique_sents)

def ensemble_best_model(summaries_dict, reference):
    """Select best single summary."""
    best_score = -1
    best_summary = ""
    
    for model_name, summary in summaries_dict.items():
        score = rouge_metric.compute(
            predictions=[summary],
            references=[reference],
            use_stemmer=True
        )["rouge2"]
        
        if score > best_score:
            best_score = score
            best_summary = summary
    
    return best_summary

def ensemble_sentence_ranking(summaries_dict):
    """Rank-based fusion."""
    sentence_positions = {}
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for pos, sent in enumerate(sents):
            normalized = sent.lower().strip()
            if normalized not in sentence_positions:
                sentence_positions[normalized] = []
            sentence_positions[normalized].append(pos)
    
    sentence_scores = {
        sent: np.mean(positions) 
        for sent, positions in sentence_positions.items()
    }
    
    ranked = sorted(sentence_scores.items(), key=lambda x: x[1])
    top_sentences = [sent for sent, _ in ranked[:15]]
    
    return " ".join(top_sentences)

# =========================================================
# GENERATE SUMMARIES FOR ALL MODELS
# =========================================================

print("\n" + "="*70)
print("🚀 GENERATING SUMMARIES WITH ALL THREE MODELS")
print("="*70)

all_model_summaries = {
    "BART": [],
    "LED": [],
    "PEGASUS": []
}

ensemble_summaries = {
    "voting": [],
    "weighted": [],
    "best_model": [],
    "ranking": []
}

references = []

for idx, item in enumerate(tqdm(test_data, desc="Processing samples")):
    judgment = item["judgment"]
    reference = item["reference_summary"]
    references.append(reference)
    
    # Generate summary with each model
    summaries_dict = {}
    
    for model_name in ["BART", "LED", "PEGASUS"]:
        summary, _ = generate_summary(model_name, judgment, reference)
        all_model_summaries[model_name].append(summary)
        summaries_dict[model_name] = summary
    
    # Apply ensemble strategies
    ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
    ensemble_summaries["weighted"].append(ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
    ensemble_summaries["best_model"].append(ensemble_best_model(summaries_dict, reference))
    ensemble_summaries["ranking"].append(ensemble_sentence_ranking(summaries_dict))

print("\n✅ All summaries generated!")

# =========================================================
# EVALUATE ALL APPROACHES
# =========================================================

print("\n" + "="*70)
print("📊 EVALUATING ALL APPROACHES")
print("="*70)

results = {}

# Evaluate individual models
for model_name in ["BART", "LED", "PEGASUS"]:
    print(f"\n  Evaluating {model_name}...")
    metrics = compute_metrics(all_model_summaries[model_name], references)
    results[model_name] = metrics
    
    print(f"  ✅ {model_name} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")

# Evaluate ensemble strategies
for strategy in ["voting", "weighted", "best_model", "ranking"]:
    print(f"\n  Evaluating Ensemble-{strategy}...")
    metrics = compute_metrics(ensemble_summaries[strategy], references)
    results[f"ensemble_{strategy}"] = metrics
    
    print(f"  ✅ Ensemble-{strategy} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")

# =========================================================
# COMPARISON TABLE
# =========================================================

print("\n" + "="*70)
print("📊 COMPREHENSIVE RESULTS COMPARISON")
print("="*70)

print(f"\n{'Approach':<20} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BERTScore F1':<12}")
print("-" * 70)

for approach_name, metrics in results.items():
    print(f"{approach_name:<20} {metrics['rouge1']:<10.4f} {metrics['rouge2']:<10.4f} "
          f"{metrics['rougeL']:<10.4f} {metrics['bertscore_f1']:<12.4f}")

# Find best approach
best_approach = max(results.items(), key=lambda x: x[1]['rouge2'])
print("\n" + "="*70)
print(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
print(f"   ROUGE-2: {best_approach[1]['rouge2']:.4f}")
print(f"   BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
print("="*70)

# =========================================================
# SAVE ALL RESULTS
# =========================================================

print("\n💾 Saving results...")

# Save individual model summaries
for model_name in ["BART", "LED", "PEGASUS"]:
    output_path = os.path.join(OUTPUT_DIR, f"{model_name.lower()}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, all_model_summaries[model_name], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save ensemble summaries
for strategy in ["voting", "weighted", "best_model", "ranking"]:
    output_path = os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, ensemble_summaries[strategy], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save complete results
complete_results = {
    "experiment": "Ensemble Summarization - BART + LED + PEGASUS (Mean Cosine Similarity)",
    "test_samples": len(test_data),
    "selection_method": "Mean Cosine Similarity",
    "mcs_k": MCS_K,
    "mcs_k_led": MCS_K_LED,
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "results": results,
    "best_approach": {
        "name": best_approach[0],
        "metrics": best_approach[1]
    }
}

results_path = os.path.join(OUTPUT_DIR, "ensemble_complete_results.json")
with open(results_path, 'w', encoding='utf8') as f:
    json.dump(complete_results, f, indent=2, ensure_ascii=False)

print(f"\n  ✅ Saved {results_path}")

# =========================================================
# GENERATE REPORT
# =========================================================

report_lines = []
report_lines.append("="*70)
report_lines.append("ENSEMBLE SUMMARIZATION EXPERIMENT")
report_lines.append("BART + LED + PEGASUS Fine-tuned Models")
report_lines.append("Selection Method: Mean Cosine Similarity")
report_lines.append("="*70)
report_lines.append("")
report_lines.append(f"Test Samples: {len(test_data)}")
report_lines.append(f"MCS Configuration: k={MCS_K} (BART/PEGASUS), k={MCS_K_LED} (LED)")
report_lines.append(f"Ensemble Weights: BART={ENSEMBLE_WEIGHTS['BART']}, LED={ENSEMBLE_WEIGHTS['LED']}, PEGASUS={ENSEMBLE_WEIGHTS['PEGASUS']}")
report_lines.append("")
report_lines.append("-"*70)
report_lines.append("INDIVIDUAL MODEL RESULTS")
report_lines.append("-"*70)

for model_name in ["BART", "LED", "PEGASUS"]:
    m = results[model_name]
    report_lines.append(f"\n{model_name}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("-"*70)
report_lines.append("ENSEMBLE RESULTS")
report_lines.append("-"*70)

for strategy in ["voting", "weighted", "best_model", "ranking"]:
    m = results[f"ensemble_{strategy}"]
    report_lines.append(f"\nEnsemble-{strategy.upper()}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("="*70)
report_lines.append(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
report_lines.append("="*70)
report_lines.append(f"ROUGE-2:      {best_approach[1]['rouge2']:.4f}")
report_lines.append(f"BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
report_lines.append("="*70)

report_text = "\n".join(report_lines)
print("\n" + report_text)

report_path = os.path.join(OUTPUT_DIR, "ensemble_report.txt")
with open(report_path, 'w', encoding='utf8') as f:
    f.write(report_text)

print(f"\n💾 Report saved to: {report_path}")

print("\n" + "="*70)
print("✅ ENSEMBLE EXPERIMENT COMPLETE")
print("="*70)
print(f"\nGenerated {len(results)} different approaches:")
print("  - 3 individual models (BART, LED, PEGASUS)")
print("  - 4 ensemble strategies (voting, weighted, best_model, ranking)")
print(f"\nBest performer: {best_approach[0]}")
print(f"Selection Method: Mean Cosine Similarity")
print("="*70)

📥 Downloading NLTK data...
🚀 Device: cuda

📋 CONFIGURATION
Selection Method: Mean Cosine Similarity
Sentences (BART/PEGASUS): 60
Sentences (LED): 200
Output directory: ./ensemble_results_mcs

📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded successfully
✅ Mean Cosine Similarity selection ready

📚 Loading all three fine-tuned models...

  [1/3] Loading BART...
  ✅ BART loaded - 406,290,432 parameters

  [2/3] Loading LED...
  ✅ LED loaded - 161,844,480 parameters

  [3/3] Loading PEGASUS...
  ✅ PEGASUS loaded - 570,797,056 parameters

✅ All models loaded successfully!

📂 Loading test data from test.json...
✅ Loaded 100 test samples

📊 Loading evaluation metrics...
✅ Metrics loaded: ROUGE, BERTScore

🚀 GENERATING SUMMARIES WITH ALL THREE MODELS


Processing samples: 100%|███████████████████████████████████████████████████████████| 100/100 [1:08:14<00:00, 40.94s/it]



✅ All summaries generated!

📊 EVALUATING ALL APPROACHES

  Evaluating BART...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ BART - ROUGE-2: 0.1727, BERTScore F1: 0.8500

  Evaluating LED...
  ✅ LED - ROUGE-2: 0.2635, BERTScore F1: 0.8534

  Evaluating PEGASUS...
  ✅ PEGASUS - ROUGE-2: 0.1832, BERTScore F1: 0.8466

  Evaluating Ensemble-voting...
  ✅ Ensemble-voting - ROUGE-2: 0.1887, BERTScore F1: 0.8394

  Evaluating Ensemble-weighted...
  ✅ Ensemble-weighted - ROUGE-2: 0.1646, BERTScore F1: 0.8382

  Evaluating Ensemble-best_model...
  ✅ Ensemble-best_model - ROUGE-2: 0.2664, BERTScore F1: 0.8547

  Evaluating Ensemble-ranking...
  ✅ Ensemble-ranking - ROUGE-2: 0.2240, BERTScore F1: 0.8337

📊 COMPREHENSIVE RESULTS COMPARISON

Approach             ROUGE-1    ROUGE-2    ROUGE-L    BERTScore F1
----------------------------------------------------------------------
BART                 0.3524     0.1727     0.2010     0.8500      
LED                  0.4980     0.2635     0.2695     0.8534      
PEGASUS              0.3724     0.1832     0.2119     0.8466      
ensemble_voting      0.3919     0.1887     